In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

# Week 1 — EDA: Credit Card Fraud Detection

**目标**：理解数据分布，识别类别不平衡的严重程度，画出欺诈 vs 正常的特征差异。

**产出**：
- 类别分布图
- V1–V5 特征在正负样本上的分布
- 时间维度上的交易密度
- 金额（Amount）的分布对比

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/transformer-anomaly

import random, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
import os, shutil, pathlib

DATA_DIR = pathlib.Path('data')
CSV = DATA_DIR / 'creditcard.csv'

if not CSV.exists():
    # 安装 kaggle CLI
    !pip install -q kaggle
    # 把 kaggle.json 放到 ~/.kaggle/
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    # 尝试从 /content 或 Drive 复制
    for src in ['/content/kaggle.json', '/content/drive/MyDrive/kaggle.json']:
        if os.path.exists(src):
            shutil.copy(src, os.path.expanduser('~/.kaggle/kaggle.json'))
            break
    else:
        raise FileNotFoundError('kaggle.json 没找到，请按上面说明上传')
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    DATA_DIR.mkdir(exist_ok=True)
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p {DATA_DIR} --unzip

print('Data ready at:', CSV)

## 1. 下载 Kaggle 数据（首次运行）

**凭证来源**：Colab Secrets（左侧 🔑）配置 `KAGGLE_USERNAME` + `KAGGLE_KEY`，bootstrap 已写入 env 变量；kaggle CLI 会自动读取。本地跑时请确保 shell 里导出同名 env 变量（或 `~/.kaggle/kaggle.json` 存在）。

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

DATA_DIR = PROJECT_ROOT / 'data'
CSV = DATA_DIR / 'creditcard.csv'

if not CSV.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !pip install -q kaggle
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p {DATA_DIR} --unzip

print('Data ready at:', CSV)

In [ ]:
df.describe().T

## 3. 类别分布（感受不平衡）

In [ ]:
counts = df['Class'].value_counts()
pct = df['Class'].value_counts(normalize=True) * 100
print('Counts:\n', counts)
print('\nPercent:\n', pct.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='Class', data=df, ax=ax[0])
ax[0].set_title('Counts (log scale)')
ax[0].set_yscale('log')
ax[1].pie(counts.values, labels=['Normal', 'Fraud'], autopct='%1.3f%%', startangle=90)
ax[1].set_title('Proportion')
plt.tight_layout()

## 4. V1–V5 分布：正常 vs 欺诈

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, ['V1', 'V2', 'V3', 'V4', 'V5']):
    sns.kdeplot(df[df.Class == 0][col], label='Normal', ax=ax, fill=True)
    sns.kdeplot(df[df.Class == 1][col], label='Fraud',  ax=ax, fill=True)
    ax.set_title(col); ax.legend()
axes.flat[-1].axis('off')
plt.tight_layout()

## 5. 金额 Amount 分布（对数尺度）

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for cls, color, label in [(0, 'C0', 'Normal'), (1, 'C3', 'Fraud')]:
    ax[0].hist(df[df.Class == cls]['Amount'], bins=80, alpha=0.6, label=label, color=color, log=True)
ax[0].set_title('Amount (log count)'); ax[0].set_xlabel('Amount'); ax[0].legend()

sns.boxplot(x='Class', y='Amount', data=df, ax=ax[1])
ax[1].set_title('Amount boxplot by class')
plt.tight_layout()

print('\nAmount 统计（按类别）:')
print(df.groupby('Class')['Amount'].describe())

## 6. 时间维度：交易密度

In [ ]:
# Time 单位是秒，总时长约 48 小时
df['Hour'] = (df['Time'] // 3600).astype(int) % 24

fig, ax = plt.subplots(figsize=(10, 4))
for cls, label in [(0, 'Normal'), (1, 'Fraud')]:
    counts = df[df.Class == cls].groupby('Hour').size()
    counts = counts / counts.sum()   # 归一化为分布
    ax.plot(counts.index, counts.values, marker='o', label=label)
ax.set_xlabel('Hour of day'); ax.set_ylabel('Density')
ax.set_title('交易小时分布：欺诈是否有时间偏好？')
ax.legend()

## 7. 本周笔记（自问自答，写给未来的自己）

1. **欺诈样本占比 ≈ 0.17%，这意味着：**
   - Accuracy 不能作为评估指标（全猜正常就 99.83%）
   - 需要 AUC-PR、Recall@FPR=0.001 这种对不平衡敏感的指标

2. **AUC-PR vs AUC-ROC 的关键差别：**
   - ROC 在极度不平衡时会过度乐观（FPR 分母很大）
   - PR 直接关心 precision/recall 折衷，更贴近业务关切

3. **V1–V5 分布观察：**
   - *（填写你看到的差异，比如 V4 在欺诈上明显右偏）*

4. **下周进入 MLP baseline 前，还需要做什么？**
   - *（填写：scaling、train/val/test 切分策略）*